# ShopStream — E-commerce Clickstream Analysis with Apache Spark

This notebook demonstrates how Spark processes a realistic e-commerce clickstream dataset.

**Dataset** — one day of synthetic traffic from an online store:
| File | Contents |
|------|----------|
| `events.csv` | Every page-view, add-to-cart, and purchase click |
| `sessions.csv` | Browser session metadata (device, country, referrer) |
| `products.csv` | Product catalogue (name, category, price) |
| `orders.csv` | Confirmed transactions |


## 1  Connect to the cluster

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Create or reuse a Spark session conncecting to our cluster (Standalone: Master, Worker, Driver)
spark = (
    SparkSession.builder
    .appName("ShopStream-Clickstream")
    .master("spark://spark-master:7077") 
    .config("spark.sql.shuffle.partitions", "4")  # keep small for local demo (2 Workers with 2 Cores)
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version) # sanity check: confirm Spark version
print("Connected to:", spark.sparkContext.master) # sanity check: confirm cluster connection

Spark version: 3.5.1
Connected to: spark://spark-master:7077


## 2  Load the CSVs

All four files are in `/data/raw/` — mounted from `./data/raw` on the host.  


In [2]:
DATA = "/data/raw"

events   = spark.read.csv(f"{DATA}/events.csv",   header=True, inferSchema=True) 
sessions = spark.read.csv(f"{DATA}/sessions.csv", header=True, inferSchema=True)
products = spark.read.csv(f"{DATA}/products.csv", header=True, inferSchema=True)

## 3  Inspect schemas

Schemas are derived from the CSV headers and sampled values (`inferSchema=True`).  
In production you would supply an explicit schema to avoid the sampling scan.

In [3]:
print("=== events ===")
events.printSchema()

print("=== sessions ===")
sessions.printSchema()

print("=== products ===")
products.printSchema()

=== events ===
root
 |-- event_id: string (nullable = true)
 |-- session_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- event_time: timestamp (nullable = true)
 |-- page_url: string (nullable = true)

=== sessions ===
root
 |-- session_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- device_type: string (nullable = true)
 |-- country: string (nullable = true)
 |-- start_time: timestamp (nullable = true)
 |-- end_time: timestamp (nullable = true)
 |-- referrer: string (nullable = true)

=== products ===
root
 |-- product_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: double (nullable = true)



In [10]:
print(f"events   : {events.count()} rows")
print(f"sessions : {sessions.count()} rows")
print(f"products : {products.count()} rows")

events   : 155 rows
sessions : 40 rows
products : 25 rows


## 4  Partitions — before and after repartitioning

When Spark reads a CSV it creates one partition per HDFS block (128 MB default) ,  
for small local files, typically one partition per file.  

We can **repartition** to spread work more evenly across workers.  

For small files this shuffle cost can be more expensive than just processing on 1 partition

In [7]:
print("Partitions per Dataframe BEFORE repartition:")
print(f"  events   : {events.rdd.getNumPartitions()}")
print(f"  sessions : {sessions.rdd.getNumPartitions()}")
print(f"  products : {products.rdd.getNumPartitions()}")

Partitions per Dataframe BEFORE repartition:
  events   : 1
  sessions : 1
  products : 1


In [9]:
# Just for showcase
# Repartition to match the number of worker cores available 
# in production let Spark manage partitions
events_rp   = events.repartition(4)
sessions_rp = sessions.repartition(4)
products_rp = products.repartition(4)

print("Partitions AFTER repartition:")
print(f"  events   : {events_rp.rdd.getNumPartitions()}")
print(f"  sessions : {sessions_rp.rdd.getNumPartitions()}")
print(f"  products : {products_rp.rdd.getNumPartitions()}")

Partitions AFTER repartition:
  events   : 4
  sessions : 4
  products : 4


## 5  Joins — enriching events with session and product context

We join the three DataFrames: events ← sessions (on session_id) ← products (on product_id). 

Each join shuffles data so matching rows end up on the same executor.

In [12]:
# Step 1 — attach session metadata to every event 
enriched = events_rp.join(sessions_rp, on="session_id", how="left") 

# Step 2 — attach product details 
enriched = enriched.join(products_rp, on="product_id", how="left")

# Keep only columns we need for analysis (drop the rest)
enriched = enriched.select(
    "event_id", "event_type", "event_time",
    "session_id", "user_id", "device_type", "country", "referrer",
    "product_id", "name", "category", "price"
).withColumnRenamed("name", "product_name")

print("Enriched schema:")
enriched.printSchema()
enriched.show(5, truncate=False)

Enriched schema:
root
 |-- event_id: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- event_time: timestamp (nullable = true)
 |-- session_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- device_type: string (nullable = true)
 |-- country: string (nullable = true)
 |-- referrer: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)

+--------+-----------+-------------------+----------+-------+-----------+-------+--------+----------+------------+-----------+-----+
|event_id|event_type |event_time         |session_id|user_id|device_type|country|referrer|product_id|product_name|category   |price|
+--------+-----------+-------------------+----------+-------+-----------+-------+--------+----------+------------+-----------+-----+
|E0068   |add_to_cart|2024-03-01 09:50:00|S1016     |U115   |desktop    |US     |goog

## 6  GroupBy aggregation — revenue and conversion by category

This is the query that showcases Spark's distributed execution model:

1. **Map phase** — each partition independently counts and sums its local rows  
2. **Shuffle phase** — partial results move across the network, grouped by `category`  
3. **Reduce phase** — final totals are merged per category on the receiving executor

The shuffle is visible in the Spark UI under the **Stages** tab — look for the "Exchange" step.

In [13]:
category_stats = (
    enriched
    .groupBy("category") #group rows by product category
    .agg( #for each category calculate total events, views, add_to_cart, 
        F.count("event_id").alias("total_events"), 
        F.sum(F.when(F.col("event_type") == "view",        1).otherwise(0)).alias("views"),
        F.sum(F.when(F.col("event_type") == "add_to_cart", 1).otherwise(0)).alias("add_to_carts"),
        F.sum(F.when(F.col("event_type") == "purchase",    1).otherwise(0)).alias("purchases"),
        F.round(
            F.sum(F.when(F.col("event_type") == "purchase", F.col("price")).otherwise(0)),
            2
        ).alias("revenue_usd"),
    )
    .withColumn( #Calculates view to pruchase conversion rate
        "conversion_rate_pct",
        F.round(F.col("purchases") / F.col("views") * 100, 1) #can be a problem if category has 0 views
    )
    .orderBy(F.col("revenue_usd").desc()) #sorts by revenue highest first
)

category_stats.show(truncate=False)

+-----------+------------+-----+------------+---------+-----------+-------------------+
|category   |total_events|views|add_to_carts|purchases|revenue_usd|conversion_rate_pct|
+-----------+------------+-----+------------+---------+-----------+-------------------+
|Electronics|57          |31   |13          |13       |1541.87    |41.9               |
|Footwear   |10          |6    |2           |2        |279.98     |33.3               |
|Kitchen    |17          |9    |5           |3        |237.97     |33.3               |
|Health     |13          |5    |4           |4        |219.96     |80.0               |
|Sports     |37          |23   |8           |6        |179.94     |26.1               |
|Accessories|11          |7    |3           |1        |44.99      |14.3               |
|Books      |6           |4    |1           |1        |22.99      |25.0               |
|Home       |4           |4    |0           |0        |0.0        |0.0                |
+-----------+------------+-----+

## 7  Top products by purchase count

In [14]:
top_products = (
    enriched
    .filter(F.col("event_type") == "purchase") #keep only purchase events
    .groupBy("product_id", "product_name", "category", "price") #group by product
    .agg(F.count("event_id").alias("purchase_count")) #how many purchases does each product have
    .orderBy(F.col("purchase_count").desc()) #sort by purchase count descending
    .limit(10)
)

top_products.show(truncate=False)

+----------+------------------------+-----------+------+--------------+
|product_id|product_name            |category   |price |purchase_count|
+----------+------------------------+-----------+------+--------------+
|P008      |Smart Watch             |Electronics|199.99|5             |
|P003      |Yoga Mat                |Sports     |34.99 |4             |
|P013      |Protein Powder          |Health     |54.99 |4             |
|P021      |Noise-Cancelling Earbuds|Electronics|99.99 |2             |
|P016      |USB Hub                 |Electronics|35.99 |2             |
|P001      |Wireless Headphones     |Electronics|79.99 |2             |
|P009      |Resistance Bands        |Sports     |19.99 |2             |
|P014      |Sunglasses              |Accessories|44.99 |1             |
|P002      |Running Shoes           |Footwear   |129.99|1             |
|P018      |Hiking Boots            |Footwear   |149.99|1             |
+----------+------------------------+-----------+------+--------

## 8  Traffic sources — sessions and purchases by referrer

In [15]:
referrer_stats = (
    enriched
    .groupBy("referrer") # which traffic sources (referrers) generate the most revenue
    .agg(
        F.countDistinct("session_id").alias("sessions"), # how many distinct visits came from that source
        F.sum(F.when(F.col("event_type") == "purchase", 1).otherwise(0)).alias("purchases"), # how many purchases
        F.round(
            F.sum(F.when(F.col("event_type") == "purchase", F.col("price")).otherwise(0)),
            2
        ).alias("revenue_usd") # total purchase value
    )
    .orderBy(F.col("revenue_usd").desc()) #sorted  by revenue
)

referrer_stats.show(truncate=False)

+---------+--------+---------+-----------+
|referrer |sessions|purchases|revenue_usd|
+---------+--------+---------+-----------+
|google   |18      |23       |1873.77    |
|direct   |5       |3        |435.97     |
|email    |6       |4        |217.96     |
|facebook |4       |0        |0.0        |
|instagram|4       |0        |0.0        |
|twitter  |3       |0        |0.0        |
+---------+--------+---------+-----------+



## 9  Query plan — `explain()` shows what Spark will actually do

`explain()` prints the **physical plan**: the exact sequence of operations Spark will execute.  


In [16]:
print("=" * 70)
print("PHYSICAL PLAN for category_stats query")
print("=" * 70)
category_stats.explain(mode="formatted")

PHYSICAL PLAN for category_stats query
== Physical Plan ==
AdaptiveSparkPlan (21)
+- Sort (20)
   +- Exchange (19)
      +- Project (18)
         +- HashAggregate (17)
            +- Exchange (16)
               +- HashAggregate (15)
                  +- Project (14)
                     +- BroadcastHashJoin LeftOuter BuildRight (13)
                        :- Project (8)
                        :  +- BroadcastHashJoin LeftOuter BuildRight (7)
                        :     :- Exchange (2)
                        :     :  +- Scan csv  (1)
                        :     +- BroadcastExchange (6)
                        :        +- Exchange (5)
                        :           +- Filter (4)
                        :              +- Scan csv  (3)
                        +- BroadcastExchange (12)
                           +- Exchange (11)
                              +- Filter (10)
                                 +- Scan csv  (9)


(1) Scan csv 
Output [4]: [event_id#17, session_id#18, 